SEQUENCE INPUT : ATCGTACGATCGGAATTCGAATTCGATCGTACGCTTAAAGCGTACGATCGGATCCGGATCCATCG



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
PSEUDOCODE
-----------
1. Start the program
2. Ask user to choose DNA input method (manual or FASTA file)
3. Read DNA sequence and convert to uppercase
4. Validate DNA sequence (A, T, G, C only)
5. Ask user for palindrome parameters:
   - minimum length
   - maximum length
   - maximum spacer length
6. Scan the DNA sequence using nested loops
7. Extract left and right sequences
8. Generate reverse complement of right sequence
9. Compare left sequence with reverse complement
10. Classify palindrome as perfect or spaced
11. Store palindrome details
12. Display formatted output
13. Ask user whether to continue or exit the program
14. End the program
"""

# Import required core Python module
import os

# ==========================================================
# DNA PALINDROME IDENTIFICATION PROGRAM
# ==========================================================

# ------------------------------
# Dictionary for DNA base pairing
# ------------------------------
# Define DNA complementary base pairs
comple_dict = {
    "A": "T",
    "T": "A",
    "G": "C",
    "C": "G"
}

# ------------------------------
# INPUT FUNCTIONS
# ------------------------------

def manual_input():
    """
    Reads DNA sequence from manual user input
    """
    # Read DNA sequence from user and convert to uppercase
    sequence = input("Enter DNA sequence: ")
    return sequence.replace(" ", "").strip().upper()


def file_input(filename):
    """
    Reads DNA sequence from a FASTA file.
    Header lines starting with '>' are ignored.
    """
    # Check if file exists
    if not os.path.exists(filename):
        print("Error! File not found.")
        return None

    sequence = ""

    # Read DNA sequence from FASTA file
    with open(filename, "r") as f:
        for line in f:
            # Skip FASTA header lines
            if line.startswith(">"):
                continue
            sequence += line.strip()

    return sequence.upper()


def input_method():
    """
    Allows user to choose DNA input method
    """
    # Ask user to choose input method
    print("\nChoose DNA input method:")
    print("1. Manual input")
    print("2. Read from FASTA file")

    choice = input("Enter 1 or 2: ")

    if choice == "1":
        return manual_input()
    elif choice == "2":
        filename = input("Enter FASTA filename/path: ")
        return file_input(filename)
    else:
        print("Invalid choice. Defaulting to manual input.")
        return manual_input()

# ------------------------------
# VALIDATION FUNCTIONS
# ------------------------------

def validate_input(sequence):
    """
    Validates that the sequence contains only A, T, G, C
    """
    # Validate DNA sequence
    for base in sequence:
        if base not in comple_dict:
            return False
    return True

# ------------------------------
# SEQUENCE PROCESSING FUNCTIONS
# ------------------------------

def complementary(sequence):
    """
    Returns the reverse complement of a DNA sequence
    """
    # Generate reverse complement
    return "".join(comple_dict[base] for base in reversed(sequence))

# ------------------------------
# PALINDROME SEARCH FUNCTIONS
# ------------------------------

def find_palindromes(sequence, min_length, max_length, max_spacer):
    """
    Finds perfect and spaced DNA palindromes
    """
    # Initialize result storage
    results = []
    seq_len = len(sequence)

    # Scan DNA sequence using nested loops
    for start in range(seq_len):
        for length in range(min_length, max_length + 1):
            for spacer in range(0, max_spacer + 1):

                # Extract left and right subsequences
                left_start = start
                left_end = start + length
                right_start = left_end + spacer
                right_end = right_start + length

                if right_end > seq_len:
                    continue

                left_seq = sequence[left_start:left_end]
                right_seq = sequence[right_start:right_end]

                # Generate reverse complement of right sequence
                rc_right = complementary(right_seq)

                # Compare left sequence with reverse complement
                if left_seq == rc_right:

                    # Classify palindrome type
                    if spacer == 0:
                        full_region = left_seq + right_seq
                        p_type = "Perfect palindrome"
                    else:
                        full_region = left_seq + ("-" * spacer) + right_seq
                        p_type = "Spaced palindrome"

                    # Store palindrome details
                    results.append({
                        "start": left_start + 1,   # 1-based indexing
                        "end": right_end,
                        "left": left_seq,
                        "right": right_seq,
                        "full_region": full_region,
                        "type": p_type
                    })

    return results

# ------------------------------
# OUTPUT FUNCTIONS
# ------------------------------

def display_results(results, seq_len):
    """
    Displays detected palindromes grouped by type
    """

    print(f"\nSequence length: {seq_len} bp")

    perfect_palindromes = []
    spaced_palindromes = []

    # Separate palindromes by type
    for p in results:
        if p["type"] == "Perfect palindrome":
            perfect_palindromes.append(p)
        else:
            spaced_palindromes.append(p)

    # -------- PERFECT PALINDROMES --------
    if len(perfect_palindromes) > 0:
        print("\nPerfect Palindromes:\n")
        print("{:<5} {:<10} {:<15} {:<15} {:<10} {:<20}".format(
            "No.", "Start", "Left Seq", "Right Seq", "End", "Full Region"
        ))

        count = 1
        for p in perfect_palindromes:
            print("{:<5} {:<10} {:<15} {:<15} {:<10} {:<20}".format(
                count,
                p["start"],
                p["left"],
                p["right"],
                p["end"],
                p["full_region"]
            ))
            count += 1
    else:
        print("\nNo perfect palindromes found.")

    # -------- SPACED PALINDROMES --------
    if len(spaced_palindromes) > 0:
        print("\nSpaced Palindromes:\n")
        print("{:<5} {:<10} {:<15} {:<15} {:<10} {:<20}".format(
            "No.", "Start", "Left Seq", "Right Seq", "End", "Full Region"
        ))

        count = 1
        for p in spaced_palindromes:
            print("{:<5} {:<10} {:<15} {:<15} {:<10} {:<20}".format(
                count,
                p["start"],
                p["left"],
                p["right"],
                p["end"],
                p["full_region"]
            ))
            count += 1
    else:
        print("\nNo spaced palindromes found.")

    print("\n------------------------------------")
    print("Total palindromes found:", len(results))


# ------------------------------
# MAIN PROGRAM
# ------------------------------

def main():
    # Step 1: Start the program
    print("--------------------------------------")
    print("DNA PALINDROME IDENTIFICATION PROGRAM")
    print("--------------------------------------")

    while True:

        # Step 2 & 3: Obtain DNA sequence
        sequence = input_method()

        if sequence is None:
            continue

        # Step 4: Validate DNA sequence
        if not validate_input(sequence):
            print("Invalid DNA sequence detected. Only A, T, G, C allowed.")
            continue

        # Step 5: Obtain palindrome parameters
        min_length = int(input("Enter minimum palindrome length: "))
        max_length = int(input("Enter maximum palindrome length: "))
        max_spacer = int(input("Enter maximum spacer length: "))

        # Step 6–11: Find palindromes
        palindromes = find_palindromes(sequence, min_length, max_length, max_spacer)

        # Step 12: Display results
        # Display message if no palindromes are found
        seq_len = len(sequence)
        if not palindromes:
            print(f"\nSequence length: {seq_len} bp")
            print("\nNo DNA palindromes were found in your sequence.")
        else:
            display_results(palindromes, len(sequence))

        # Step 13: Ask to continue or exit the program
        choice = input("\nContinue? (Y to continue, N to exit): ").upper()
        if choice != "Y":
            print("Program terminated. Thank you.")
            break


# Step 14: End the program
if __name__ == "__main__":
    main()


--------------------------------------
DNA PALINDROME IDENTIFICATION PROGRAM
--------------------------------------

Choose DNA input method:
1. Manual input
2. Read from FASTA file
Enter 1 or 2: 1
Enter DNA sequence: ATCGTACGATCGGAATTCGAATTCGATCGTACGCTTAAAGCGTACGATCGGATCCGGATCCATCG
Enter minimum palindrome length: 6
Enter maximum palindrome length: 15
Enter maximum spacer length: 3

Sequence length: 65 bp

Perfect Palindromes:

No.   Start      Left Seq        Right Seq       End        Full Region         
1     13         GAATTC          GAATTC          24         GAATTCGAATTC        
2     50         GGATCC          GGATCC          61         GGATCCGGATCC        

Spaced Palindromes:

No.   Start      Left Seq        Right Seq       End        Full Region         
1     24         CGATCGTACGCT    AGCGTACGATCG    50         CGATCGTACGCT---AGCGTACGATCG
2     24         CGATCGTACGCTT   AAGCGTACGATCG   50         CGATCGTACGCTT-AAGCGTACGATCG
3     25         GATCGTACGCT     AGCGTACGATC 